# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnushkaWagh03/Flyrank-ML-IIntern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am treating the **Refresh / Content Opportunity Scoring** lane as a **ranking problem**. The goal is not simply to classify every page as “refresh” or “do not refresh”; the practical decision is to determine **which existing content items should be reviewed first**. The output will therefore be a priority score that allows content reviewers to rank pages from higher to lower refresh opportunity. Ranking fits the real action because editorial time is limited, so the most useful result is a shortlist of the pages that deserve attention first.

In [2]:
import pandas as pd

path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)

print(f"Loaded {len(df)} rows and {df.shape[1]} columns.")

print("ML task type: Ranking")
print("Decision: Which content items should be reviewed first?")
print("Output: A priority score used to rank content items.")

Loaded 30000 rows and 44 columns.
ML task type: Ranking
Decision: Which content items should be reviewed first?
Output: A priority score used to rank content items.


## 2. Target or proxy

For the starter dataset, I will use `is_declining_proxy` as a temporary target for exploration. It is defined as `trend_direction == "down"`, so it represents an observed current-window condition rather than a true future outcome. This makes it useful for understanding the starter workflow, but it is not an ideal final target because the label is derived from the current trend measurement. For the capstone, I would prefer a future-looking observed outcome, such as whether a page experiences a measurable decline during a later time window. The final ranking should therefore be evaluated against an outcome that occurs after the feature window, rather than simply reproducing a current rule.

In [3]:
df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Target/proxy column created: is_declining_proxy")
print(df["is_declining_proxy"].value_counts().sort_index())

Target/proxy column created: is_declining_proxy
is_declining_proxy
0    13738
1    16262
Name: count, dtype: int64


## 3. Success metric

My primary success metric will be **Precision@K**, where K represents the number of highest-ranked pages a reviewer can realistically inspect. Precision@K fits this decision because the purpose of the system is to put useful candidates near the top of the review queue. A high Precision@K means that a large proportion of the pages selected for early review actually match the outcome being investigated. I will compare the ranking approach against a simple baseline before deciding whether ML adds enough value to justify its complexity.

In [4]:
K = 50

print(f"Primary success metric: Precision@{K}")
print("Reason: reviewers have limited time, so the quality of the top-ranked items matters most.")

Primary success metric: Precision@50
Reason: reviewers have limited time, so the quality of the top-ranked items matters most.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content item**. Each row represents one piece of existing content for a client, with its observed search and engagement measurements over the available 90-day window. The model will assign a score to each content item, and those scores will be used to create the review ranking. `content_id` and `client_id` are identifiers used for grouping and splitting, not predictive features.

In [5]:
# Show the actual unit of analysis.
# One row = one pseudonymized content item.

columns_to_show = [
    "content_id",
    "client_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_direction",
    "is_declining_proxy"
]

unit_df = df[columns_to_show].head(10)

display(unit_df)

,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,0.03,8.5,147,down,1
6,content_9a34b442b552,client_8722616204,20,0,1,0.00,7.0,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,28,0.06,21.2,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,0.09,46.0,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,3,0.16,4.9,257,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could identify obvious candidates, for example pages with a downward trend and low engagement, but the refresh decision can depend on several signals at the same time: search demand, impressions, clicks, CTR, sessions, average position, content age, content size, and recent activity. A single hand-written threshold may miss combinations where several moderate signals together indicate an opportunity. ML is worth testing because it can combine these signals into a consistent ranking and potentially improve the quality of the top-K review queue. However, ML does not automatically beat a rule: I will only consider it useful if it performs better than a simple baseline on the pre-declared Precision@K metric using leakage-aware validation.

In [6]:
candidate_features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count",
    "char_count"
]

available_features = [c for c in candidate_features if c in df.columns]

print("Candidate observable features:")
print(available_features)
print(f"Number of candidate signals: {len(available_features)}")
print("\nIDs and trend fields are excluded from the candidate feature list.")

Candidate observable features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'content_age_days', 'word_count', 'char_count']
Number of candidate signals: 8

IDs and trend fields are excluded from the candidate feature list.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.